# Retrieval, reranking and query understanding on SciFact

A small retrieval pipeline over the [SciFact](https://huggingface.co/datasets/BeIR/scifact) benchmark (BEIR): 5,183 scientific abstracts, claim-style queries, and human relevance judgments (qrels). The notebook builds the pipeline one stage at a time, measures each change against a baseline, and analyses where it fails.

**Pipeline:** bi-encoder (`all-MiniLM-L6-v2`) retrieves the top 50 candidates, then a cross-encoder reranks them. Query understanding (intent, entities) and query expansion are layered on and evaluated.

**Protocol.** Choices (PRF settings, rules, routing) are made on the **train** qrels. The **test** qrels (300 queries) are used for the headline comparison. Reusable code lives in `src/`; this notebook is the analysis.

> Reported numbers in the markdown cells come from the original Colab run. Re-running the cells reproduces them up to model/library versions.

## 0. Setup

In [ ]:
import json, random, sys
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
from datasets import load_dataset
from sentence_transformers import CrossEncoder, SentenceTransformer

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.build_index import build_index
from src.evaluation import (compare_runs, evaluate_run, mean_recall, ndcg_at_k,
                            precision_at_k, recall_at_k, reciprocal_rank)
from src.expansion import expand_with_entities, prf_query_vector
from src.query_understanding import (ALIAS_TABLE, detect_intent, link_entities, link_entities_fuzzy)

INDEX = ROOT / "data" / "index"
if not (INDEX / "doc_vectors.npy").exists():
    build_index(INDEX)          # embeds the corpus once (~minutes on CPU, fast on a Colab GPU)

doc_vectors = np.load(INDEX / "doc_vectors.npy")
doc_ids = json.loads((INDEX / "doc_ids.json").read_text())
doc_texts = json.loads((INDEX / "doc_texts.json").read_text())
query_lookup = json.loads((INDEX / "queries.json").read_text())      # query id -> text
doc_text_lookup = dict(zip(doc_ids, doc_texts))

qrels_ds = load_dataset("BeIR/scifact-qrels")
def build_qrels(split):
    d = defaultdict(set)
    for q, doc in zip(split["query-id"], split["corpus-id"]):
        d[str(q)].add(str(doc))
    return dict(d)

qrels_train, qrels_test = build_qrels(qrels_ds["train"]), build_qrels(qrels_ds["test"])
train_qids, test_qids = sorted(qrels_train), sorted(qrels_test)

model = SentenceTransformer("all-MiniLM-L6-v2")
print(doc_vectors.shape, len(doc_ids), len(query_lookup), len(train_qids), len(test_qids))

## 1. Data checks

In [ ]:
# Every qrels row is a relevant document (score 1); unlisted documents count as not relevant.
print("train scores:", set(qrels_ds["train"]["score"]), "| test scores:", set(qrels_ds["test"]["score"]))

per_query = sorted(len(v) for v in qrels_test.values())
print("relevant docs per test query (min / median / max):", per_query[0], per_query[len(per_query)//2], per_query[-1])

# ids connect across the three tables, and train / test do not overlap
test_docs = {d for v in qrels_test.values() for d in v}
print("test queries missing from queries:", len(set(qrels_test) - set(query_lookup)))
print("test docs missing from corpus:    ", len(test_docs - set(doc_ids)))
print("train/test query overlap:         ", len(set(qrels_train) & set(qrels_test)))

words = sorted(len(t.split()) for t in doc_texts)
print("doc words (min / median / max):", words[0], words[len(words)//2], words[-1])
print("MiniLM max tokens:", model.max_seq_length)

Checks passed: 300 test queries, 1 to 5 relevant documents per query (median 1), no missing ids, no train/test overlap. **Known limitation:** the median document is 204 words (roughly 270-300 tokens) but MiniLM reads only 256 tokens, so more than half of the documents are truncated. Accepted for the baseline; chunking is listed under next steps.

## 2. Metric implementations

`MRR@10`, `precision@k`, `nDCG@10` and `recall@k` are implemented by hand in `src/evaluation.py` and unit-tested against hand calculations (`tests/test_evaluation.py`), including a query with two relevant documents, where nDCG's ideal DCG behaves differently. Precision@10 is low by design here: most queries have one or two relevant documents, so it cannot approach 1.0.

In [ ]:
# Sanity check: relevant doc at rank 5 -> RR 0.2, P@10 0.1, nDCG 1/log2(6) = 0.387
fake = list("abcdefghij")
print(reciprocal_rank(fake, {"e"}), precision_at_k(fake, {"e"}, 10), round(ndcg_at_k(fake, {"e"}, 10), 3))

## 3. Baseline: bi-encoder only

In [ ]:
test_q_vecs = model.encode([query_lookup[q] for q in test_qids], batch_size=64, normalize_embeddings=True)

def dense_run(qids, vecs, n=50):
    run = {}
    for q, v in zip(qids, vecs):
        top = np.argsort(-(doc_vectors @ v))[:n]
        run[q] = [doc_ids[p] for p in top]
    return run

candidates_test = dense_run(test_qids, test_q_vecs, n=50)
baseline = evaluate_run(candidates_test, qrels_test)
print("baseline:", {k: round(v, 4) for k, v in baseline.items()})
for k in (10, 20, 50):
    print(f"Recall@{k}:", round(mean_recall(candidates_test, qrels_test, k), 4))

| System | MRR@10 | P@10 | nDCG@10 |
|---|---|---|---|
| Bi-encoder only | 0.6047 | 0.0883 | 0.6451 |

Recall of the bi-encoder: @10 = 0.7833, @20 = 0.8373, **@50 = 0.8920**. About 11% of correct documents never reach the reranker, so recall@50 is the ceiling for anything built on top of these candidates. The gap between recall@10 and recall@50 is the room a reranker has to work with.

## 4. Reranking

A bi-encoder embeds the query and each document separately (fast, cannot see them together). A cross-encoder reads the query and one document as a single input (more accurate, too slow for the whole corpus). So: bi-encoder narrows 5,183 documents to 50, cross-encoder reorders those 50.

In [ ]:
def rerank_all(ce, qids, cands, batch_size=16):
    run = {}
    for q in qids:
        pairs = [(query_lookup[q], doc_text_lookup[d]) for d in cands[q]]
        scores = ce.predict(pairs, batch_size=batch_size)
        run[q] = [cands[q][i] for i in np.argsort(-scores)]
    return run

ce_marco = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", max_length=512)
ce_bge = CrossEncoder("BAAI/bge-reranker-base", max_length=512)

rerank_marco_test = rerank_all(ce_marco, test_qids, candidates_test)
rerank_bge_test = rerank_all(ce_bge, test_qids, candidates_test)

for name, run in [("bi-encoder", candidates_test), ("+ MS MARCO MiniLM", rerank_marco_test), ("+ bge-reranker-base", rerank_bge_test)]:
    print(f"{name:22}", {k: round(v, 4) for k, v in evaluate_run(run, qrels_test).items()})

| System | MRR@10 | P@10 | nDCG@10 |
|---|---|---|---|
| Bi-encoder only | 0.6047 | 0.0883 | 0.6451 |
| + MS MARCO MiniLM reranker | 0.6628 | 0.0923 | 0.6936 |
| + bge-reranker-base | 0.6754 | 0.0927 | 0.7049 |

Adding a reranker was the big gain (about +6 MRR, +5 nDCG). Swapping one reranker for the other added about +1 point, at roughly 10x the model size (about 1 GB vs 90 MB). The next levers are the 11% of correct documents missing from the top 50, and truncation.

### 4.1 Spread, not just the mean

In [ ]:
for name, run in [("MS MARCO", rerank_marco_test), ("bge", rerank_bge_test)]:
    c = compare_runs(candidates_test, run, qrels_test)
    print(f"{name}: winners {c['winners']} | ties {c['ties']} | losers {c['losers']}")

cmp_marco = compare_runs(candidates_test, rerank_marco_test, qrels_test)
for i in np.argsort(cmp_marco["diff"])[:5]:                      # five biggest drops
    q = cmp_marco["qids"][i]
    print(q, "RR", round(reciprocal_rank(candidates_test[q], qrels_test[q], 10), 3), "->",
          round(reciprocal_rank(rerank_marco_test[q], qrels_test[q], 10), 3), "|", query_lookup[q])

MS MARCO reranker: 72 queries improved, 189 unchanged, 39 got worse. bge-reranker-base: 73 / 189 / 38. Winners outnumber losers almost 2 to 1, but the average hides real damage: the five worst regressions were all at **rank 1** before reranking (RR 1.0 dropping to between 0.1 and 0.25), and four of the five are negated or directional claims ("does not cause any adverse autoimmune events", "precipitates adverse autoimmune events", "increases inflammation-related fibrosis", "does not affect sporulation efficiency").

## 5. Error analysis: does negation hurt the reranker?

**Hypothesis (from five examples).** The MS MARCO reranker was trained on questions and answer passages. SciFact queries are *claims* that documents support or refute, so it may focus on topic overlap and miss negation and direction cues. Five hand-picked worst cases are a hunch, not evidence, so test it on all 300 queries with crude keyword groups.

In [ ]:
neg_words = ["not", "no ", "lack", "without", "never", "fails", "exclusively", "only"]
is_neg = np.array([any(w in query_lookup[q].lower() for w in neg_words) for q in test_qids])

d_marco, d_bge = cmp_marco["diff"], compare_runs(candidates_test, rerank_bge_test, qrels_test)["diff"]
for name, d in [("MS MARCO", d_marco), ("bge", d_bge)]:
    print(f"{name}: negation (n={is_neg.sum()}) {d[is_neg].mean():+.4f} | others (n={(~is_neg).sum()}) {d[~is_neg].mean():+.4f}")

| Group (test) | MS MARCO: avg change in RR | bge: avg change in RR |
|---|---|---|
| Negation-style (40) | -0.027 | +0.006 |
| Others (260) | +0.071 | +0.081 |

Negation was the only group where the first reranker made things worse on average. Direction claims (106 queries) showed at most mild dampening (+0.038 vs +0.069), and the "absolutes" group had 3 queries, too few to say anything.

**Replication on train failed.** Routing negated queries (per `detect_intent`) back to the bi-encoder was evaluated on the 809 train queries (section 9). On the 74 negated train queries the reranker was still clearly better than the bi-encoder, so the negation effect was **not robust**: the test group had only 40 queries and the keyword tag is crude.

## 6. Intent detection

Three intents: `negated`, `direction` (increase/decrease/more likely...), `other`. I hand-labeled 50 random train queries (`data/intent_labels_set_a.json`) and compared a keyword/regex detector with a learned one (MiniLM embeddings + logistic regression, stratified 5-fold CV).

In [ ]:
labels_a = json.loads((ROOT / "data" / "intent_labels_set_a.json").read_text())
lab_qids = list(labels_a)
y = np.array([labels_a[q] for q in lab_qids])

rule_preds = np.array([detect_intent(query_lookup[q]) for q in lab_qids])
print("rules accuracy (tuned on these same 50!):", (rule_preds == y).mean())

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import StratifiedKFold, cross_val_predict

X = model.encode([query_lookup[q] for q in lab_qids], normalize_embeddings=True)
cv_preds = cross_val_predict(LogisticRegression(max_iter=1000), X, y,
                             cv=StratifiedKFold(5, shuffle=True, random_state=42))
print("learned accuracy (5-fold CV):", (cv_preds == y).mean())
print(classification_report(y, cv_preds, zero_division=0))

| Detector | Accuracy on the 50 labeled queries |
|---|---|
| Rules, first version | 0.80 (10 misses) |
| Rules, after fixing misses | 0.94 (3 misses) |
| Learned (MiniLM + LR, 5-fold CV) | 0.64 |

**Read the 0.94 carefully:** the rules were fixed by inspecting exactly these 50 queries, so it is an optimistic, tuned-on-the-test-set number. The fair comparison is the original 0.80 against the learned 0.64, and the honest number for the final rules needs fresh labels (a second 50-query set was sampled but never labeled or scored). The learned model never predicted `negated` correctly (0.00 recall, 5 examples): embeddings capture topic far more than grammar, and 50 labels is too few. That does not show rules beat learned models in general. These labels are defined by cue words, which favors rules.

Fixes that came from the misses: add direction vocabulary ("more likely", "accelerates", "represses", "worsens"), and use word boundaries so "tumor suppressor" is not read as "suppress". Two labels (queries 925 and 86) were changed from `direction` to `other`, because they made no increase/decrease claim; the labels in the data file are the corrected ones.

## 7. Entity extraction and linking

General spaCy (`en_core_web_sm`) was trained on web text and is unusable on biomedical claims: it missed p53, qSOFA, and H3K9me3, and mislabeled gene names (PTEN, SRF as `ORG`, SCNT as `GPE`). Its *spans* are sometimes useful candidates even when the labels are wrong; noun chunks recover more candidates, with noise.

So linking uses a **hand-written alias table** (canonical name to aliases) with whole-token exact matching and optional fuzzy matching (`rapidfuzz`) for near-misses. A real system would import HGNC / UMLS / MeSH instead.

In [ ]:
for q in ["Binding of p53 to enhancer regions (p53BERs) regulates cell cycle arrest through p53-dependent eRNAs.",
          "Removal of H3K9me3 by ectopic expression of other H3K9 demethylases improves SCNT reprogramming.",
          "The PPR MDA5 has twenty N-terminal CARD domains."]:
    print(link_entities(q), "<-", q[:70])

for t in ["Binding of p-53 to enhancer regions", "Removal of H3K9 me3 improves reprogramming", "Marmoset brains", "The pten gene"]:
    print(link_entities_fuzzy(t), "<-", t)

Known weaknesses: the table was built from ~10 queries, so coverage is tiny (section 8); "PPR" is ambiguous in general; the table maps aliases to canonical names but does not resolve related entities (H3K9 vs H3K9me3 are separate entries on purpose).

## 8. Query expansion (tuned on train)

**Pseudo-relevance feedback (PRF).** Blend the query vector with the mean of the top-m first-pass documents, re-search. **Entity-alias expansion.** Append the aliases of linked entities to the query text.

In [ ]:
tr_vecs = model.encode([query_lookup[q] for q in train_qids], batch_size=64, normalize_embeddings=True)

def eval_prf(m, alpha):
    rr, nd, rec = [], [], []
    for q, v in zip(train_qids, tr_vecs):
        qv = prf_query_vector(v, doc_vectors, m=m, alpha=alpha)
        ranked = [doc_ids[p] for p in np.argsort(-(doc_vectors @ qv))[:50]]
        rel = qrels_train[q]
        rr.append(reciprocal_rank(ranked, rel, 10)); nd.append(ndcg_at_k(ranked, rel, 10)); rec.append(recall_at_k(ranked, rel, 50))
    return [round(float(np.mean(x)), 4) for x in (rr, nd, rec)]

print("baseline (m=0)", eval_prf(0, 1.0), " # [MRR@10, nDCG@10, Recall@50]")
for m in (3, 5):
    for alpha in (0.5, 0.7, 0.9):
        print(f"m={m} alpha={alpha}", eval_prf(m, alpha))

PRF on train (MRR@10, nDCG@10, Recall@50): baseline 0.6261 / 0.6602 / 0.8903. The best setting (m=3, alpha=0.9) gave 0.6248 / 0.6601 / **0.8977**: a gain of about 0.7 points of recall@50 and nothing on MRR or nDCG. Low alpha (0.5) hurt. **Verdict:** a candidate-stage tool with little benefit here.

In [ ]:
touched = [q for q in train_qids if expand_with_entities(query_lookup[q]) != query_lookup[q]]
print("train queries touched by alias expansion:", len(touched), "of", len(train_qids))

def search_text(text, n=50):
    v = model.encode(text, normalize_embeddings=True)
    return [doc_ids[p] for p in np.argsort(-(doc_vectors @ v))[:n]]

for q in touched:
    rel = qrels_train[q]
    a, b = search_text(query_lookup[q]), search_text(expand_with_entities(query_lookup[q]))
    print(q, "RR", round(reciprocal_rank(a, rel, 10), 2), "->", round(reciprocal_rank(b, rel, 10), 2),
          "| R@50", round(recall_at_k(a, rel, 50), 2), "->", round(recall_at_k(b, rel, 50), 2))

Alias expansion matched only **14 of 809** train queries (1.7%). On those 14: MRR@10 0.3794 to 0.4602, nDCG@10 0.4250 to 0.5194, recall@50 0.8571 to 0.9286. 3 queries improved, 11 were unchanged, none got worse. But one query (eRNAs, "enhancer rna" pulled the correct document from outside the top 10 to rank 1) drives most of the gain, so this is fragile evidence. **Verdict:** the mechanism works for vocabulary mismatch; coverage is too low to move the overall metrics. A larger imported entity table is the fix, not a better method.

## 9. Connecting intent to retrieval (routing)

If the reranker hurts negated claims, route them: keep the bi-encoder's ranking for `negated` queries, use the reranker for the rest. Design choices are made on **train**, so this builds train candidates and reranked lists first (about 4 minutes on CPU).

In [ ]:
train_cands = dense_run(train_qids, tr_vecs, n=50)
train_reranked = rerank_all(ce_marco, train_qids, train_cands, batch_size=32)

routed = {q: (train_cands[q] if detect_intent(query_lookup[q]) == "negated" else train_reranked[q]) for q in train_qids}
neg_q = [q for q in train_qids if detect_intent(query_lookup[q]) == "negated"]

for name, run in [("bi-encoder only", train_cands), ("reranker only", train_reranked), ("intent routed", routed)]:
    r = evaluate_run(run, qrels_train)
    print(f"{name:16} MRR@10 {r['MRR@10']:.4f}  nDCG@10 {r['nDCG@10']:.4f}")
print("negated queries:", len(neg_q))
for name, run in [("bi-encoder", train_cands), ("reranker", train_reranked)]:
    r = evaluate_run({q: run[q] for q in neg_q}, qrels_train)
    print(f"negated only, {name:10} MRR@10 {r['MRR@10']:.4f}  nDCG@10 {r['nDCG@10']:.4f}")

| System (809 train queries) | MRR@10 | nDCG@10 |
|---|---|---|
| Bi-encoder only | 0.6261 | 0.6602 |
| Reranker only | 0.6787 | 0.7109 |
| Intent routed | 0.6732 | 0.7064 |

On the 74 negated train queries the reranker still beats the bi-encoder (MRR@10 0.6783 vs 0.6185; nDCG@10 0.7070 vs 0.6581). Routing sent them to the weaker system, so it scored slightly *below* reranker-only. The hypothesis formed on test errors **did not replicate** on larger, independent data. That is a legitimate result, not a failure: the hard routing rule is not worth shipping.

## 10. Summary

| System | MRR@10 | P@10 | nDCG@10 |
|---|---|---|---|
| Bi-encoder only (test, 300 queries) | 0.6047 | 0.0883 | 0.6451 |
| + MS MARCO MiniLM reranker | 0.6628 | 0.0923 | 0.6936 |
| + bge-reranker-base | 0.6754 | 0.0927 | 0.7049 |

**Takeaways**
- A baseline comes first; the reranker stage is the big gain, the choice between these two rerankers is small.
- Recall@50 caps the reranker: 11% of correct documents never reach it.
- Check winners / ties / losers and slice by query type; the average hid per-query regressions at rank 1.
- A hypothesis from five examples needs fresh data. Mine did not replicate.
- Rules beat a learned model on tiny, cue-word-defined labels; small samples (14 queries, one big winner) make fragile evidence.

**Limitations**
- Test queries were read during error analysis, so the test numbers are slightly optimistic. (The original notebook also drew a dev/test split of all queries for later work but never used it; it is omitted here.)
- Documents are truncated at 256 tokens for the bi-encoder; no chunking was tried.
- Intent labels come from one person on a small sample; the final rules were tuned on those same 50 queries and have not been scored on fresh labels.
- The alias table is hand-written and tiny. Relevant-but-unlabeled documents count as misses in qrels-based evaluation.
- The learned intent model was not improved (more labels, TF-IDF n-grams, cue features).

**Next steps:** chunking or a longer-context embedding model, reciprocal rank fusion, HyDE, an imported entity table, and a fresh labeled intent set.